In [1]:
import os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import json, nibabel as nib

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torch.optim.lr_scheduler import CosineAnnealingLR

# CAMBIO 1: Usar ResNet en lugar de DenseNet
from monai.networks.nets import resnet18
from monai.transforms import (
    Compose, EnsureType, 
    RandRotate90, RandFlip, RandAffine,
    RandGaussianNoise, RandGaussianSmooth,
    NormalizeIntensity
)

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_auc_score

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
from collections import defaultdict

2025-12-21 11:34:44.495014: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766316884.514877 2775897 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766316884.520719 2775897 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766316884.535097 2775897 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766316884.535137 2775897 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766316884.535139 2775897 computation_placer.cc:177] computation placer alr

In [2]:
#config
device = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS = 50  # CAMBIO 2: Más épocas con early stopping
PATIENCE = 15  # CAMBIO 3: Early stopping

RUN_TAG = time.strftime("%Y%m%d-%H%M%S")
RESULTS_CSV = f"resultados_210pacientes_resnet18_improved_{RUN_TAG}.csv"

In [3]:
def append_rows_to_csv(rows, csv_path=RESULTS_CSV):
    """Escribe filas (lista de dicts) al CSV, añadiendo cabecera solo si no existe."""
    df_rows = pd.DataFrame(rows)
    file_exists = os.path.exists(csv_path)
    df_rows.to_csv(csv_path, mode="a", header=not file_exists, index=False)


#leer datos
df = pd.read_csv("./../clinical_data/clinical_data.csv", na_values="NaN")
labels_dict_numeric = {
    k: 1 if str(v).strip().lower() in ['sí', 'si', 'yes', '1', 'true'] else 0
    for k, v in dict(zip(df['patient_id'], df['Complicación'])).items()
}

In [4]:
# CAMBIO 4: Calcular estadísticas del dataset para normalización
def compute_dataset_stats(dataset, num_samples=50):
    """Calcula media y std del dataset para normalización"""
    loader = DataLoader(dataset, batch_size=1, shuffle=True)
    means, stds = [], []
    
    for i, (vol, _) in enumerate(loader):
        if i >= num_samples:
            break
        vol_np = vol.numpy()
        means.append(vol_np[vol_np > 0].mean())  # Solo pixels no-cero
        stds.append(vol_np[vol_np > 0].std())
    
    return np.mean(means), np.mean(stds)

In [5]:
#gradcam (mantener como estaba)
def get_last_conv_layer(model):
    # Para ResNet, la última capa conv está en layer4
    if hasattr(model, 'layer4'):
        return model.layer4[-1].conv2
    # Fallback para otros modelos
    for layer in reversed(list(model.children())):
        if isinstance(layer, nn.Conv3d):
            return layer
    raise ValueError("No se encontró una capa Conv3d válida en el modelo")


In [6]:
def apply_gradcam(model, volume, label, device, output_base_dir, example_id=""):
    model.train()
    target_layers = [get_last_conv_layer(model)]
    cam = GradCAM(model=model, target_layers=target_layers)

    input_tensor = volume.to(device)
    targets = [ClassifierOutputTarget(label)]

    with torch.enable_grad():
        grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0]

    for idx in range(0, grayscale_cam.shape[0], max(1, grayscale_cam.shape[0] // 6)):
        plt.figure(figsize=(6, 6))
        img = input_tensor.cpu().numpy()[0, 0, idx]
        norm_img = (img - img.min()) / (img.max() - img.min() + 1e-8)
        heatmap = grayscale_cam[idx]
        rgb_img = np.repeat(norm_img[..., np.newaxis], 3, axis=-1)
        result = show_cam_on_image(rgb_img.astype(np.float32), heatmap, use_rgb=True)

        os.makedirs(output_base_dir, exist_ok=True)
        path = os.path.join(output_base_dir, f"{example_id}_slice_{idx}.png")
        plt.imsave(path, result)
        plt.close()


def visualizar_gradcams(model, dataset, val_idx, device, output_base_dir):
    print("Generando ejemplos de Grad-CAM...")
    correctos = defaultdict(list)
    incorrectos = defaultdict(list)

    loader = DataLoader(Subset(dataset, val_idx), batch_size=1, shuffle=False)
    model.eval()

    for i, (vol, label) in enumerate(loader):
        vol = vol.to(device)
        real = label.item()
        pred = torch.argmax(model(vol), dim=1).item()

        patient_id = dataset.patients[val_idx[i]]

        if pred == real:
            correctos[real].append( (vol, patient_id, pred) )
        else:
            incorrectos[real].append( (vol, patient_id, pred) )

    # Generar todos los ejemplos
    for clase in [0, 1]:
        for (v, pid, pred) in correctos[clase]:
            example_id = f"label{clase}_correcto_pred{pred}_pid{pid}"
            apply_gradcam(model, v, clase, device, output_base_dir, example_id=example_id)

        for (v, pid, pred) in incorrectos[clase]:
            example_id = f"label{clase}_fallo_pred{pred}_pid{pid}"
            apply_gradcam(model, v, clase, device, output_base_dir, example_id=example_id)


In [7]:
class LungCTMaskedNPYDataset(Dataset):
    """Dataset con un solo canal enfatizando el nódulo y/o las vessels"""
    def __init__(
        self,
        root_npy_dir,
        labels_dict,
        patients=None,
        transform=None,
        emphasize_nodule=False,
        nodule_gain=0.1,
        emphasize_vessels=0.1,        # ### VESSELS
        vessel_gain=0.1                 # ### VESSELS
    ):
        self.root = root_npy_dir
        self.images_dir = os.path.join(root_npy_dir, "images")
        self.lung_dir   = os.path.join(root_npy_dir, "masks_lung")
        self.nod_dir    = os.path.join(root_npy_dir, "masks_nodule")
        self.vess_dir   = os.path.join(root_npy_dir, "masks_vessels")  # ### VESSELS
        self.labels = labels_dict
        self.patients = patients if patients is not None else list(labels_dict.keys())
        self.transform = transform
        self.emphasize_nodule = emphasize_nodule
        self.nodule_gain = float(nodule_gain)
        self.emphasize_vessels = emphasize_vessels          # ### VESSELS
        self.vessel_gain = float(vessel_gain)               # ### VESSELS

        # Filtra por ficheros existentes (incluyendo vessels)
        self.patients = [
            pid for pid in self.patients
            if all(os.path.exists(os.path.join(d, f"{pid}.npy"))
                   for d in [self.images_dir, self.lung_dir, self.nod_dir, self.vess_dir])
        ]

    def __len__(self): 
        return len(self.patients)

    def __getitem__(self, idx):
        pid  = self.patients[idx]
        img  = np.load(os.path.join(self.images_dir, f"{pid}.npy")).astype(np.float32)
        lung = np.load(os.path.join(self.lung_dir,   f"{pid}.npy")).astype(np.float32)
        nod  = np.load(os.path.join(self.nod_dir,    f"{pid}.npy")).astype(np.float32)
        vess = np.load(os.path.join(self.vess_dir,   f"{pid}.npy")).astype(np.float32)  # ### VESSELS

        lung = (lung > 0.5).astype(np.float32)
        nod  = (nod  > 0.5).astype(np.float32)
        vess = (vess > 0.5).astype(np.float32)                                           # ### VESSELS

        # Enmascara solo intrapulmonar
        img_masked = img * lung

        # Realza el nódulo si está activado
        if self.emphasize_nodule and self.nodule_gain > 0.0:
            factor_nod = 1.0 + self.nodule_gain * nod
            img_masked = img_masked * factor_nod
            img_masked = np.clip(img_masked, 0.0, 1.0)

        # Realza las vessels si está activado
        if self.emphasize_vessels and self.vessel_gain > 0.0:
            factor_vess = 1.0 + self.vessel_gain * vess
            img_masked = img_masked * factor_vess
            img_masked = np.clip(img_masked, 0.0, 1.0)

        vol = img_masked[None, ...].astype(np.float32)

        if self.transform:
            vol = self.transform(vol)

        y = torch.tensor(self.labels[pid], dtype=torch.long)
        return vol, y


In [8]:
# CAMBIO 6: Transforms con data augmentation agresivo
def get_transforms(is_train=True, mean=0.5, std=0.25):
    """Retorna transforms apropiados para train o val/test"""
    if is_train:
        return Compose([
            RandRotate90(prob=0.5, spatial_axes=(0, 2)),
            RandFlip(prob=0.5, spatial_axis=0),
            RandFlip(prob=0.5, spatial_axis=1),
            RandFlip(prob=0.5, spatial_axis=2),
            RandAffine(
                prob=0.3,
                rotate_range=[0.15, 0.15, 0.15],
                scale_range=[0.15, 0.15, 0.15],
                mode='bilinear',
                padding_mode='zeros'
            ),
            RandGaussianNoise(prob=0.2, std=0.01),
            RandGaussianSmooth(prob=0.2, sigma_x=(0.5, 1.0)),
            NormalizeIntensity(subtrahend=mean, divisor=std),
            EnsureType()
        ])
    else:
        return Compose([
            NormalizeIntensity(subtrahend=mean, divisor=std),
            EnsureType()
        ])

In [9]:
# CAMBIO 7: Modelo ResNet18 en lugar de DenseNet
def build_model(dropout_prob=0.3):
    """Construye ResNet18 3D con dropout"""
    model = resnet18(
        spatial_dims=3,
        n_input_channels=1,
        num_classes=2
    )
    
    # Añadir dropout antes de la capa final
    if dropout_prob > 0:
        model.fc = nn.Sequential(
            nn.Dropout(p=dropout_prob),
            model.fc
        )
    
    return model

In [10]:
# CAMBIO 9: Función para graficar training history
def plot_training_history(history, save_path):
    """Grafica las curvas de entrenamiento"""
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Loss
    axes[0].plot(history['train_loss'], label='Train Loss')
    axes[0].plot(history['val_loss'], label='Val Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Loss Evolution')
    axes[0].legend()
    axes[0].grid(True)
    
    # Accuracy
    axes[1].plot(history['train_acc'], label='Train Acc')
    axes[1].plot(history['val_acc'], label='Val Acc')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_title('Accuracy Evolution')
    axes[1].legend()
    axes[1].grid(True)
    
    # G-Mean
    axes[2].plot(history['train_gmean'], label='Train G-Mean')
    axes[2].plot(history['val_gmean'], label='Val G-Mean')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('G-Mean')
    axes[2].set_title('G-Mean Evolution')
    axes[2].legend()
    axes[2].grid(True)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()


#metricas (mantener igual)
def calcular_metricas_binarias(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    if cm.shape != (2, 2):
        return 0, 0, 0
    TN, FP, FN, TP = cm.ravel()
    tpr = TP / (TP + FN) if (TP + FN) else 0
    tnr = TN / (TN + FP) if (TN + FP) else 0
    gmean = np.sqrt(tpr * tnr)
    return tpr, tnr, gmean

def evaluar_modelo(model, data_loader, device):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            probs = torch.softmax(outputs, dim=1)[:, 1]
            preds = torch.argmax(outputs, dim=1).cpu().numpy()

            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    tpr, tnr, gmean = calcular_metricas_binarias(all_labels, all_preds)

    try:
        auc = roc_auc_score(all_labels, all_probs)
    except ValueError:
        auc = float("nan")

    return acc, f1, tpr, tnr, gmean, auc

In [11]:
# CAMBIO 8: Entrenamiento con early stopping y class weights
def train_model_with_internal_validation(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    scheduler,
    device,
    epochs,
    save_path,
    patience=15
):
    best_val_gmean = 0.0
    epochs_no_improve = 0
    history = {
        'train_loss': [], 'val_loss': [],
        'train_acc': [], 'val_acc': [],
        'train_gmean': [], 'val_gmean': []
    }

    for epoch in range(epochs):
        # TRAIN
        model.train()
        running_loss = 0.0
        all_preds, all_labels = [], []

        print(f"\n Epoch {epoch+1}/{epochs}")
        for inputs, labels in tqdm(train_loader, desc="Training"):
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

        train_loss = running_loss / len(train_loader)
        acc = accuracy_score(all_labels, all_preds)
        f1 = f1_score(all_labels, all_preds, zero_division=0)
        tpr, tnr, gmean = calcular_metricas_binarias(all_labels, all_preds)
        
        history['train_loss'].append(train_loss)
        history['train_acc'].append(acc)
        history['train_gmean'].append(gmean)
        
        print(f" Train — Loss: {train_loss:.4f} | Acc: {acc:.4f} | F1: {f1:.4f} | G-Mean: {gmean:.4f}")

        # VALIDATION
        model.eval()
        val_loss = 0.0
        val_preds, val_labels = [], []

        with torch.no_grad():
            for val_inputs, val_labels_batch in val_loader:
                val_inputs, val_labels_batch = val_inputs.to(device), val_labels_batch.to(device)
                val_outputs = model(val_inputs)
                batch_loss = criterion(val_outputs, val_labels_batch)
                val_loss += batch_loss.item()

                val_preds_batch = torch.argmax(val_outputs, dim=1).cpu().numpy()
                val_preds.extend(val_preds_batch)
                val_labels.extend(val_labels_batch.cpu().numpy())

        val_loss = val_loss / len(val_loader)
        val_acc = accuracy_score(val_labels, val_preds)
        val_f1 = f1_score(val_labels, val_preds, zero_division=0)
        val_tpr, val_tnr, val_gmean = calcular_metricas_binarias(val_labels, val_preds)

        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_gmean'].append(val_gmean)

        print(f" Val   — Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | F1: {val_f1:.4f} | G-Mean: {val_gmean:.4f}")

        scheduler.step()

        # CAMBIO: Guardar SIEMPRE en época 0, luego solo si mejora
        if epoch == 0 or val_gmean > best_val_gmean:
            if epoch == 0 and val_gmean <= 0:
                print(f" ⚠️ Época 1: G-Mean={val_gmean:.4f}, guardando modelo de seguridad")
            else:
                print(f" ✓ Mejor modelo guardado (G-Mean: {val_gmean:.4f})")
            
            best_val_gmean = max(val_gmean, best_val_gmean)  # Asegurar que mejore
            torch.save(model.state_dict(), save_path)
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            print(f" ✗ Sin mejora ({epochs_no_improve}/{patience})")

        if epochs_no_improve >= patience:
            print(f"\n Early stopping activado en época {epoch+1}")
            break

    # Graficar curvas de aprendizaje
    plot_training_history(history, save_path.replace('.pth', '_history.png'))
    
    return history


In [12]:
# CAMBIO 10: Cross-validation mejorado con class weights
def cross_validate(
    model_class,
    dataset,
    batch_size,
    learning_rate,
    preprocessed_dir,
    k=5,
    device='cuda',
    epochs=50,
    save_path_prefix="modelo_resnet18_210pacientes_",
    weight_decay=1e-5,
    dropout_prob=0.3,
    seed=42,
    nodule_gain=0.1, 
    vessel_gain=0.1
):
    results = []
    labels = [dataset.labels[pid] for pid in dataset.patients]
    
    # CAMBIO: Calcular class weights
    class_counts = np.bincount(labels)
    class_weights = len(labels) / (len(class_counts) * class_counts)
    class_weights_tensor = torch.FloatTensor(class_weights).to(device)
    print(f"Class weights: {class_weights} (counts: {class_counts})")
    
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=seed)

    for fold, (train_val_idx, test_idx) in enumerate(skf.split(dataset.patients, labels)):
        print("\n" + "="*60)
        print(f" Fold {fold+1}/{k} | BS={batch_size} | LR={learning_rate}")
        print("="*60)

        # Internal split 80/20
        internal_labels = [dataset.labels[dataset.patients[i]] for i in train_val_idx]
        train_indices, val_indices = train_test_split(
            train_val_idx,
            test_size=0.2,
            stratify=internal_labels,
            random_state=seed
        )

        # CAMBIO: Crear datasets con transforms diferentes para train/val
        # Calcular stats solo con datos de train
        train_subset_temp = Subset(dataset, train_indices)
        mean_val, std_val = compute_dataset_stats(train_subset_temp, num_samples=30)
        print(f"Dataset stats - Mean: {mean_val:.3f}, Std: {std_val:.3f}")
        
        # Recrear dataset con transforms apropiados
        dataset.transform = get_transforms(is_train=True, mean=mean_val, std=std_val)
        train_subset = Subset(dataset, train_indices)
        
        dataset.transform = get_transforms(is_train=False, mean=mean_val, std=std_val)
        val_subset = Subset(dataset, val_indices)
        test_subset = Subset(dataset, test_idx)

        train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, 
                                 num_workers=4, pin_memory=True)
        val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, 
                               num_workers=4, pin_memory=True)
        test_loader = DataLoader(test_subset, batch_size=batch_size, shuffle=False, 
                                num_workers=4, pin_memory=True)

        model = model_class(dropout_prob).to(device)
        optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
        criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
        scheduler = CosineAnnealingLR(optimizer, T_max=epochs)
        
        save_path = f"{save_path_prefix}_bs{batch_size}_lr{learning_rate}_fold{fold+1}.pth"

        # Entrenar con early stopping
        history = train_model_with_internal_validation(
            model, train_loader, val_loader, criterion, optimizer, scheduler,
            device, epochs, save_path, patience=PATIENCE
        )

        # Evaluación final
        model.load_state_dict(torch.load(save_path))
        val_acc, val_f1, val_tpr, val_tnr, val_gmean, val_auc = evaluar_modelo(model, val_loader, device)
        test_acc, test_f1, test_tpr, test_tnr, test_gmean, test_auc = evaluar_modelo(model, test_loader, device)

        # Guardar resultados
        row_val = {
            "fold": fold + 1, "set": "VALIDATION",
            "accuracy": val_acc, "f1": val_f1, "tpr": val_tpr, "tnr": val_tnr,
            "gmean": val_gmean, "auc": val_auc,
            "batch_size": batch_size, "learning_rate": learning_rate,
            "weight_decay": weight_decay, "dropout_prob": dropout_prob, 
            "seed": seed, "nodule_gain": nodule_gain, "vessel_gain": vessel_gain,
            "preprocessed_dir": preprocessed_dir
        }
        row_test = {
            "fold": fold + 1, "set": "TEST",
            "accuracy": test_acc, "f1": test_f1, "tpr": test_tpr, "tnr": test_tnr,
            "gmean": test_gmean, "auc": test_auc,
            "batch_size": batch_size, "learning_rate": learning_rate,
            "weight_decay": weight_decay, "dropout_prob": dropout_prob, 
            "seed": seed, "nodule_gain": nodule_gain, "vessel_gain": vessel_gain,
            "preprocessed_dir": preprocessed_dir
        }

        append_rows_to_csv([row_val, row_test])
        results.extend([row_val, row_test])

        # GradCAM
        gradcam_output_dir = (
            f"gradcam_outputs_resnet18/"
            f"{preprocessed_dir}_ng{nodule_gain}_vg{vessel_gain}/fold_{fold+1}"
        )

        visualizar_gradcams(model, dataset, test_idx, device, gradcam_output_dir)

    results_df = pd.DataFrame(results)

    # Calcular medias
    for split in ["VALIDATION", "TEST"]:
        means = results_df[results_df["set"] == split][["accuracy","f1","tpr","tnr","gmean","auc"]].mean()
        mean_row = {
            "fold": "MEAN", "set": split,
            "accuracy": means["accuracy"], "f1": means["f1"],
            "tpr": means["tpr"], "tnr": means["tnr"],
            "gmean": means["gmean"], "auc": means["auc"],
            "batch_size": batch_size, "learning_rate": learning_rate,
            "weight_decay": weight_decay, "dropout_prob": dropout_prob,
            "seed": seed, "nodule_gain": nodule_gain, "vessel_gain": vessel_gain,
            "preprocessed_dir": preprocessed_dir
        }
        append_rows_to_csv([mean_row])
        results_df = pd.concat([results_df, pd.DataFrame([mean_row])], ignore_index=True)

    return results_df



In [13]:
# CAMBIO 11: Grid search simplificado
if __name__ == "__main__":
    preprocessing_dirs = [
        # "resize_cube64_hu_m600_1500",   # Empezar con el más pequeño
        # "resize_cube128_hu_m600_1500",
        "resize_small_hu_m300_1400"
    ]

    # Búsqueda más enfocada
    learning_rates_to_try = [1e-4]
    weight_decays_to_try = [1e-5]
    dropout_probs_to_try = [0.5]
    nodule_gains_to_try = [0.0]  # Probar con y sin énfasis
    vessel_gains_to_try = [0.5]
    seeds_to_try = [8]

    def bs_list_for(prep: str):
        if "cube64" in prep:
            return [4]
        if "cube128" in prep:
            return [4]
        return [2]

    all_results = []

    for seed in seeds_to_try:
        print("\n" + "#"*60)
        print(f"=== SEED: {seed} ===")
        print("#"*60)

        for prep in preprocessing_dirs:
            print("\n" + "#"*60)
            print(f"=== PREPROCESADO: {prep} | SEED={seed} ===")
            print("#"*60)

            base_dir = f"/mnt/homeGPU/mcribilles/tfm/volumenes_preprocesados/{prep}/npy"
            
            for nodule_gain in nodule_gains_to_try:
                for vessel_gain in vessel_gains_to_try:
                    # tags para nombres de fichero (0.3 -> 0p3)
                    ng_tag = str(nodule_gain).replace(".", "p")
                    vg_tag = str(vessel_gain).replace(".", "p")

                    dataset = LungCTMaskedNPYDataset(
                        root_npy_dir=base_dir,
                        labels_dict=labels_dict_numeric,
                        transform=None,  # Se asigna en cross_validate
                        emphasize_nodule=(nodule_gain > 0),
                        nodule_gain=nodule_gain,
                        emphasize_vessels=(vessel_gain > 0),
                        vessel_gain=vessel_gain
                    )
                    
                    print(f"[INFO] Dataset: {len(dataset)} pacientes | "
                        f"Nodule gain: {nodule_gain} | Vessel gain: {vessel_gain}")
                    
                    if len(dataset) == 0:
                        print(f"[WARN] Dataset vacío, saltando...")
                        continue

                    for bs in bs_list_for(prep):
                        for lr in learning_rates_to_try:
                            for wd in weight_decays_to_try:
                                for dropout in dropout_probs_to_try:
                                    print(f"\n{'='*60}")
                                    print(
                                        f"Config: BS={bs}, LR={lr}, WD={wd}, Drop={dropout}, "
                                        f"NoduleGain={nodule_gain}, VesselGain={vessel_gain}"
                                    )
                                    print(f"{'='*60}")

                                    df_result = cross_validate(
                                        model_class=build_model,
                                        dataset=dataset,
                                        batch_size=bs,
                                        learning_rate=lr,
                                        preprocessed_dir=prep,
                                        k=5,
                                        device=device,
                                        epochs=EPOCHS,
                                        save_path_prefix=(
                                            f"resnet18_{prep}_bs{bs}_lr{lr}_wd{wd}_drop{dropout}_"
                                            f"ng{ng_tag}_vg{vg_tag}_seed{seed}"
                                        ),
                                        weight_decay=wd,
                                        dropout_prob=dropout,
                                        seed=seed,
                                        nodule_gain=nodule_gain,
                                        vessel_gain=vessel_gain            # ### VESSELS
                                    )

                                    all_results.append(df_result)


    # Guardar resultados finales
    final_results = pd.concat(all_results, ignore_index=True)
    final_results.to_csv("resultados_finales_resnet18_improved_210pacientes.csv", index=False)
    print("\n✓ Resultados guardados en 'resultados_finales_resnet18_improved_210pacientes.csv'")


############################################################
=== SEED: 8 ===
############################################################

############################################################
=== PREPROCESADO: resize_small_hu_m300_1400 | SEED=8 ===
############################################################
[INFO] Dataset: 200 pacientes | Nodule gain: 0.0 | Vessel gain: 0.5

Config: BS=2, LR=0.0001, WD=1e-05, Drop=0.5, NoduleGain=0.0, VesselGain=0.5
Class weights: [0.98039216 1.02040816] (counts: [102  98])

 Fold 1/5 | BS=2 | LR=0.0001
Dataset stats - Mean: 0.190, Std: 0.174


This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.



 Epoch 1/50


Training: 100%|██████████| 64/64 [00:46<00:00,  1.36it/s]

 Train — Loss: 0.7411 | Acc: 0.6094 | F1: 0.5968 | G-Mean: 0.6089


...

 Val   — Loss: nan | Acc: 0.5000 | F1: 0.0000 | G-Mean: 0.0000
 ✗ Sin mejora (15/15)

 Early stopping activado en época 16


You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what t

Generando ejemplos de Grad-CAM...


invalid value encountered in cast
invalid value encountered in cast



✓ Resultados guardados en 'resultados_finales_resnet18_improved_210pacientes.csv'
